### Script to Easily Parse Results

### Imports

In [1]:
import os
import numpy as np

### Key Variables to Determine WHich Files to Parse

In [2]:
MODEL_VERS = 'vit_b16_ep100_ctxv1' # 'vit_b16_ep100_ctxv1' or 'rn50_ep100'
VERS_OF_INFERENCE = 'tgt' # For DollarStreet: src, tgt, as, am, af - which logs to look at
LIST_OF_TEST_CASES = [#'dollarstreet_default',                                   # which subfolders to look at 
                     # 'tgt_llm_ensemble_dollarstreet',
                     # 'tgt_in_country_ensemble_dollarstreet',
                      'tgt_llm_and_in_country_ensemble_dollarstreet'] 

ROOT_PATH_REPO = '/ihome/akovashka/krb115/projects/geo_knowledge_prompting_proj/GeoKnowledgePrompting/'
SUBFOLDER_START = 'output/all/'
EXTRA_APPEND = '' # check paths to see if this is needed 

In [3]:
# This determines which results we are looking at 
if VERS_OF_INFERENCE == "src":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'train_all/' + EXTRA_APPEND 
elif VERS_OF_INFERENCE == "as":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/' + EXTRA_APPEND + 'full_as_dollarstreet/' 
elif VERS_OF_INFERENCE == "am":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START +'test_all/'  + EXTRA_APPEND + 'full_am_dollarstreet/'
elif VERS_OF_INFERENCE == "af":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_af_dollarstreet/'
elif VERS_OF_INFERENCE == "eu":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_eu_dollarstreet/'
elif VERS_OF_INFERENCE == "spec_src":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'train_all/'  + EXTRA_APPEND 
elif VERS_OF_INFERENCE == "spec_tgt":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/am_'  + EXTRA_APPEND + 'full_tgt_dollarstreet/'
elif VERS_OF_INFERENCE == "sub":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'subset_tgt_dollarstreet/'
elif VERS_OF_INFERENCE == "high_econ":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_high_econ_dollarstreet/'
elif VERS_OF_INFERENCE == "low_econ":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_low_econ_dollarstreet/'
elif VERS_OF_INFERENCE == "medium_econ":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_medium_econ_dollarstreet/'
elif VERS_OF_INFERENCE == "Bolivia":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_bolivia_dollarstreet/'
elif VERS_OF_INFERENCE == "Brazil":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_brazil_dollarstreet/'
elif VERS_OF_INFERENCE == "Canada":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_canada_dollarstreet/'
elif VERS_OF_INFERENCE == "Colombia":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_colombia_dollarstreet/'
elif VERS_OF_INFERENCE == "Guatemala":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_guatemala_dollarstreet/'
elif VERS_OF_INFERENCE == "Haiti":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_haiti_dollarstreet/'
elif VERS_OF_INFERENCE == "Mexico":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_mexico_dollarstreet/'
elif VERS_OF_INFERENCE == "Peru":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_peru_dollarstreet/'
elif VERS_OF_INFERENCE == "United States":
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_unitedstates_dollarstreet/'
else:
    OUT_PATH = ROOT_PATH_REPO + SUBFOLDER_START + 'test_all/'  + EXTRA_APPEND + 'full_tgt_dollarstreet/'
print(OUT_PATH)

/ihome/akovashka/krb115/projects/geo_knowledge_prompting_proj/GeoKnowledgePrompting/output/all/test_all/full_tgt_dollarstreet/


### Make Dictionary of Metrics

In [4]:
# Function to parse classwise accuracies 
def parse_line(line):
    class_id = int(line.split('* class: ')[1].split('(')[0])
    class_name = line.split('(')[1].split(')')[0]
    num_correct = int(line.split('correct: ')[1].split('\t')[0])
    total = int(line.split('total: ')[1].split('\t')[0])
    overall_acc = num_correct / total
    return class_id, class_name, num_correct, total, overall_acc

In [7]:
# Define a dictionary to store data from parsing logs
dict_case_and_weight_to_data = dict()

# Loop through each test case 
for case in LIST_OF_TEST_CASES:
    print(case)
    gen_path = os.path.join(OUT_PATH, case)
    
    # Define current tests to look at 
    # For DollarStreet Default, we want to look at CoOp, CoCoOp, etc. 
    # GeoKgCoOp used with ensemble options
    if case == 'am_dollarstreet_default' or case == 'dollarstreet_default' or case == 'geonet' or case == 'subset_tgt_dollarstreet':
        curr_modes = ['CoOp', 'CoCoOp', 'KgCoOp', 'GeoKnowledgePrompting'] # change this later
    else:
        curr_modes =  ['GeoKnowledgePrompting']

    # Loop through current modes
    for mode in curr_modes: 
    
        # Which weights should we gather 
        if mode == 'GeoKnowledgePrompting':
            tests = ['shots_16_2.0', 'shots_16_4.0', 'shots_16_6.0', 'shots_16_8.0', 'shots_16_10.0'] 
        elif mode == 'KgCoOp':
            tests = ['shots_16_2.0', 'shots_16_4.0', 'shots_16_6.0', 'shots_16_8.0', 'shots_16_10.0']
        elif mode == 'CoOp' or mode == 'CoCoOp':
            tests = ['shots_16_0.0']
            if case == 'subset_tgt_dollarstreet':
                tests += ['shots_1_0.0', 'shots_2_0.0', 'shots_4_0.0', 'shots_6_0.0', 'shots_8_0.0','shots_10_0.0','shots_12_0.0','shots_14_0.0','shots_16_0.0']
        # Loop through relevant test cases 
        for weight_vers in tests:

            # Make sure current path exists 
            path_to_look_at = os.path.join(gen_path, weight_vers + '/' + mode)
            if not os.path.isdir(path_to_look_at):
                continue
            print('\t' + path_to_look_at)
            
            print(path_to_look_at)
            # Add current experiment to dictionary along with placeholders for metrics 
            curr_key = case + '_' + weight_vers + '_' + mode
            dict_case_and_weight_to_data[curr_key] = dict()
            dict_case_and_weight_to_data[curr_key]['F1'] = []
            dict_case_and_weight_to_data[curr_key]['Per-Class Avg Acc'] = []
            dict_case_and_weight_to_data[curr_key]['Overall Acc'] = []

            # Loop through 3 seeds 
            path_to_seed = os.path.join(gen_path, weight_vers + '/' + mode + '/' + MODEL_VERS + '/seed')
            if mode == 'CoCoOp' and case == 'geonet':
                path_to_seed = os.path.join(gen_path, weight_vers + '/' + mode + '/' + 'vit_b16_ep10_ctxv1' + '/seed')
                
            for seed in range(1,4):

                # If not an existing path, ignore 
                if not os.path.isdir(path_to_seed + str(seed)):
                    continue

                # Update seed in dictionary and create log path
                dict_case_and_weight_to_data[curr_key][seed] = dict()
                path_to_log = path_to_seed + str(seed) + '/log.txt'

                # Load current log and parse information
                with open(path_to_log, 'r') as f:
                    lines = f.readlines()
                    for l in lines:
                        if l.startswith('* accuracy'):
                            dict_case_and_weight_to_data[curr_key]['Overall Acc'].append(float(l.strip().split('* accuracy: ')[1].strip('%')))    
                        if l.startswith('* macro_f1'):
                            dict_case_and_weight_to_data[curr_key]['F1'].append(float(l.strip().split('* macro_f1: ')[1].strip('%')))   
                        if l.startswith('* average'):
                            dict_case_and_weight_to_data[curr_key]['Per-Class Avg Acc'].append(float(l.strip().split('* average: ')[1].strip('%')))    
                        if l.startswith('* class'):
                            class_id, class_name, num_correct, total, overall_acc = parse_line(l.strip())
                            dict_case_and_weight_to_data[curr_key][seed][class_name] = overall_acc

tgt_llm_and_in_country_ensemble_dollarstreet
	/ihome/akovashka/krb115/projects/geo_knowledge_prompting_proj/GeoKnowledgePrompting/output/all/test_all/full_tgt_dollarstreet/tgt_llm_and_in_country_ensemble_dollarstreet/shots_16_4.0/GeoKnowledgePrompting
/ihome/akovashka/krb115/projects/geo_knowledge_prompting_proj/GeoKnowledgePrompting/output/all/test_all/full_tgt_dollarstreet/tgt_llm_and_in_country_ensemble_dollarstreet/shots_16_4.0/GeoKnowledgePrompting


### Produce Results for Excel Spreadsheet

In [8]:
WEIGHT_OF_INTEREST = '4.0'  # which weight value to look at (enter as string like '4.0')
print(dict_case_and_weight_to_data.keys())

files_in_order = [
    'dollarstreet_default_shots_16_0.0_CoOp',
    'dollarstreet_default_shots_16_0.0_CoCoOp',
    'dollarstreet_default_shots_16_' + WEIGHT_OF_INTEREST + '_KgCoOp',
    'tgt_in_country_ensemble_dollarstreet_shots_16_' + WEIGHT_OF_INTEREST + '_GeoKnowledgePrompting',
    'tgt_llm_ensemble_dollarstreet_shots_16_' + WEIGHT_OF_INTEREST + '_GeoKnowledgePrompting',
    'tgt_llm_and_in_country_ensemble_dollarstreet_shots_16_' + WEIGHT_OF_INTEREST + '_GeoKnowledgePrompting',
]

dict_keys(['tgt_llm_and_in_country_ensemble_dollarstreet_shots_16_4.0_GeoKnowledgePrompting'])


In [9]:
# Print results in order of files 
# Leave placeholder value for percent change 
for key in files_in_order:
    if key in dict_case_and_weight_to_data:
        curr_overall_acc_list = dict_case_and_weight_to_data[key]['Overall Acc']
        curr_f1_list = dict_case_and_weight_to_data[key]['F1']
        curr_per_class_acc_list = dict_case_and_weight_to_data[key]['Per-Class Avg Acc']
        print(*([key] + curr_overall_acc_list + [np.mean(curr_overall_acc_list), 0.00] + curr_f1_list
             + [np.mean(curr_f1_list), 0.00] + curr_per_class_acc_list + [np.mean(curr_per_class_acc_list), 0.00]))

tgt_llm_and_in_country_ensemble_dollarstreet_shots_16_4.0_GeoKnowledgePrompting 61.94 61.48 61.87 61.76333333333333 0.0 59.6 59.44 59.3 59.44666666666666 0.0 64.02 63.52 63.45 63.663333333333334 0.0


### Get Classwise Performance

In [10]:
# Compute average accuracy for each class 
def get_avg_acc_by_cls_dict(key, dict_case_and_weight_to_data):
    acc_dict = dict()
    for cls_name in dict_case_and_weight_to_data[key][1]:
        acc_1 = dict_case_and_weight_to_data[key][1][cls_name]
        acc_2 = dict_case_and_weight_to_data[key][2][cls_name]
        acc_3 = dict_case_and_weight_to_data[key][3][cls_name]
        avg_acc_for_cls = (acc_1 + acc_2 + acc_3) / 3
        acc_dict[cls_name] = avg_acc_for_cls
    return acc_dict

# Compute what the classes with a default accuracy are 
def get_baseline_classes_lt_threshold_dict(baseline_acc_dict, threshold_list=[0.40, 0.60, 0.80, 1.00]):
    classes_lt_threshold_dict = dict()
    for t in threshold_list:
        classes_lt_threshold_dict[t] = []
        for cls_name in baseline_acc_dict:
            if t == 1.00:
                if baseline_acc_dict[cls_name] <= t:
                    classes_lt_threshold_dict[t].append(cls_name)
            else:
                if baseline_acc_dict[cls_name] < t:
                    classes_lt_threshold_dict[t].append(cls_name)
    return classes_lt_threshold_dict

# Get average accuracy of only the classes lt a threshold - compute over thresholds 
def get_avg_acc_of_classes_lt_threshold(curr_avg_acc_dict, baseline_classes_lt_threshold_dict):
    dict_thresh_to_avg_acc_across_spec_classes = dict()
    for t in baseline_classes_lt_threshold_dict:
        curr_accs = []
        #print(len(baseline_classes_lt_threshold_dict[t]))
        for cls_name in baseline_classes_lt_threshold_dict[t]:
            curr_accs.append(curr_avg_acc_dict[cls_name])
        total_acc = sum(curr_accs)/len(curr_accs)
        dict_thresh_to_avg_acc_across_spec_classes[t] = total_acc
    return dict_thresh_to_avg_acc_across_spec_classes